In [2]:
import numpy as np
import json
import os

class TwoLevel_AHP_Model:
    def __init__(self):
        # --- 第一層：主維度 ---
        self.main_criteria = [
            "Return_Main", 
            "Risk_Main", 
            "Cost_Main", 
            "Liquidity_Main", 
            "Diversity_Main", 
            "Sentiment_Main"
        ]
        
        # --- 第二層：子特徵 ---
        self.sub_criteria = {
            "Return_Main": ["Return_CAGR", "Return_Div"],
            "Risk_Main": ["Risk_Vol", "Risk_MaxDD"],
            "Liquidity_Main": ["Liq_Volume", "Liq_AUM"],
            "Cost_Main": ["Cost_ExpRatio"],       # 單一特徵，權重為 1
            "Diversity_Main": ["Div_Score"],      # 單一特徵，權重為 1
            "Sentiment_Main": ["FinBERT_score"]   # 單一特徵，權重為 1
        }
        
        # 定義不同矩陣大小對應的 Random Index (RI) 查表值
        self.RI_dict = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24}

    def _solve_matrix(self, comparisons, n):
        """通用的 AHP 矩陣求解器"""
        if n == 1:
            return np.array([1.0]), 0.0 # 單一特徵不需比較
            
        num_comparisons = (n * (n - 1)) // 2
        if len(comparisons) != num_comparisons:
            raise ValueError(f"維度為 {n} 的矩陣需要 {num_comparisons} 個成對比較值。")

        matrix = np.ones((n, n))
        idx = 0
        for i in range(n):
            for j in range(i + 1, n):
                val = comparisons[idx]
                matrix[i, j] = val
                matrix[j, i] = 1.0 / val
                idx += 1
                
        eigenvalues, eigenvectors = np.linalg.eig(matrix)
        max_idx = np.argmax(np.real(eigenvalues))
        max_eigenvalue = np.real(eigenvalues[max_idx])
        eigenvector = np.real(eigenvectors[:, max_idx])
        
        weights = eigenvector / np.sum(eigenvector)
        
        # 當 n<=2 時，CR 理論上恆為 0，且 RI 為 0 無法相除，因此直接回傳 CR=0
        if n <= 2:
            CR = 0.0
        else:
            CI = (max_eigenvalue - n) / (n - 1)
            CR = CI / self.RI_dict[n]
            
        return weights, CR

    def calculate_global_weights(self, user_inputs):
        """
        傳入使用者的問卷結果 (包含第一層與需要比較的第二層)。
        計算出最終 9 個子特徵的全局權重。
        """
        print("\n🚀 啟動兩層級 AHP (Two-Level AHP) 運算...")
        
        # 1. 求解第一層主維度權重
        print("\n[第一層：主維度求解]")
        main_weights, main_cr = self._solve_matrix(user_inputs["Main"], len(self.main_criteria))
        
        main_weight_dict = {crit: weight for crit, weight in zip(self.main_criteria, main_weights)}
        for crit, weight in main_weight_dict.items():
            print(f"  - {crit}: {weight*100:.2f}%")
        print(f"  >> 一致性比率 (CR): {main_cr:.4f}")
        
        if main_cr > 0.1:
            print("❌ 警告：主維度問卷存在邏輯矛盾 (CR > 0.1)！")
            
        # 2. 求解第二層子特徵權重並計算全局權重
        print("\n[第二層：子特徵局部權重與全局權重]")
        global_weights = {}
        
        for main_crit in self.main_criteria:
            subs = self.sub_criteria[main_crit]
            main_w = main_weight_dict[main_crit]
            
            # 如果該維度只有 1 個子特徵，局部權重為 1.0
            if len(subs) == 1:
                sub_name = subs[0]
                global_weights[sub_name] = main_w * 1.0
                print(f"  - {sub_name} (單一特徵) -> 全局權重: {global_weights[sub_name]*100:.2f}%")
                continue
                
            # 如果有多個子特徵，需要求解次矩陣
            sub_comparisons = user_inputs["Sub"].get(main_crit, [])
            local_weights, sub_cr = self._solve_matrix(sub_comparisons, len(subs))
            
            for i, sub_name in enumerate(subs):
                global_w = main_w * local_weights[i]
                global_weights[sub_name] = global_w
                print(f"  - {sub_name} (局部 {local_weights[i]*100:.1f}%) -> 全局權重: {global_w*100:.2f}%")
                
        return global_weights, main_cr


# ==========================================
# 互動式問卷前處理器 (Questionnaire Preprocessor)
# ==========================================
def ask_question(question, options):
    print(f"\n{question}")
    for key, value in options.items():
        print(f"  ({key}) {value['text']}")
    
    while True:
        ans = input("請輸入你的選擇: ").strip().upper()
        if ans in options:
            return options[ans]['scores']
        print("❌ 輸入無效，請重新輸入。")

def score_diff_to_ahp(diff):
    """將絕對分數的差異映射為 AHP 的 1~9 比較尺度"""
    if diff == 0: return 1.0
    elif diff == 1: return 3.0
    elif diff == 2: return 5.0
    elif diff == 3: return 7.0
    elif diff >= 4: return 9.0
    else: return 1.0 / score_diff_to_ahp(abs(diff))

def build_user_simulation():
    """透過 CLI 問卷收集使用者偏好，並轉換為類別所需的字典格式"""
    print("="*60)
    print(" 🧠 智能理財引擎：投資屬性與偏好分析問卷")
    print("="*60)
    
    # --- 1. 收集主維度分數 ---
    scores = {
        'Return_Main': 3.0, 'Risk_Main': 3.0, 'Cost_Main': 3.0,
        'Liquidity_Main': 3.0, 'Diversity_Main': 3.0, 'Sentiment_Main': 3.0
    }

    q1 = "Q1. 若遇到全球股災，你最多能忍受多少帳面虧損？"
    opt1 = {
        'A': {'text': '絕對不能虧損 (極度厭惡風險)', 'scores': {'Risk_Main': +2, 'Return_Main': -2}},
        'B': {'text': '虧損 10% 以內', 'scores': {'Risk_Main': +1, 'Return_Main': -1}},
        'C': {'text': '虧損 20% 左右', 'scores': {'Risk_Main': 0, 'Return_Main': 0}},
        'D': {'text': '虧損 30% 也能接受，長期會漲回來', 'scores': {'Risk_Main': -1, 'Return_Main': +1}},
        'E': {'text': '腰斬也不怕，危機就是轉機 (極度追求報酬)', 'scores': {'Risk_Main': -2, 'Return_Main': +2}}
    }
    
    q2 = "Q2. 購買高單價商品時，你的消費習慣是？"
    opt2 = {
        'A': {'text': '極度精打細算，到處比價找折扣', 'scores': {'Cost_Main': +2}},
        'B': {'text': '稍微比價，不花太多時間', 'scores': {'Cost_Main': +1}},
        'C': {'text': '東西好稍微貴一點無所謂', 'scores': {'Cost_Main': -1}},
        'D': {'text': '只看品牌和頂規，不在乎價差', 'scores': {'Cost_Main': -2}}
    }

    q3 = "Q3. 你未來 1 到 3 年內臨時需要動用這筆資金的機率？"
    opt3 = {
        'A': {'text': '極高，可能隨時變現救急', 'scores': {'Liquidity_Main': +2}},
        'B': {'text': '有點可能，需保留彈性', 'scores': {'Liquidity_Main': +1}},
        'C': {'text': '機率很低', 'scores': {'Liquidity_Main': -1}},
        'D': {'text': '絕對不會動用，鎖死 10 年也沒差', 'scores': {'Liquidity_Main': -2}}
    }

    q4 = "Q4. 若要經營一門生意，你偏好？"
    opt4 = {
        'A': {'text': '複合式商場，什麼都賣以分散風險', 'scores': {'Diversity_Main': +2}},
        'B': {'text': '幾樣主力商品搭配配件', 'scores': {'Diversity_Main': +1}},
        'C': {'text': '專賣店，專攻利基市場', 'scores': {'Diversity_Main': -1}},
        'D': {'text': '極致單一產品，要麼大賺要麼倒閉', 'scores': {'Diversity_Main': -2}}
    }

    q5 = "Q5. 當新聞大肆報導 AI 科技爆發時，你的反應是？"
    opt5 = {
        'A': {'text': '立刻跟進，跟著資金流賺趨勢財', 'scores': {'Sentiment_Main': +2}},
        'B': {'text': '適度參與，不全盤投入', 'scores': {'Sentiment_Main': +1}},
        'C': {'text': '不受影響，只看基本面', 'scores': {'Sentiment_Main': -1}},
        'D': {'text': '新聞越熱我越不敢碰', 'scores': {'Sentiment_Main': -2}}
    }

    for q, opt in [(q1, opt1), (q2, opt2), (q3, opt3), (q4, opt4), (q5, opt5)]:
        ans_scores = ask_question(q, opt)
        for key, val in ans_scores.items():
            scores[key] += val

    # 確保絕對分數介於 1~5
    for key in scores:
        scores[key] = max(1.0, min(5.0, scores[key]))

    # 將 6 個維度的絕對分數，轉換為 15 個成對比較值 (順序必須對齊類別中的雙迴圈)
    keys = ["Return_Main", "Risk_Main", "Cost_Main", "Liquidity_Main", "Diversity_Main", "Sentiment_Main"]
    main_comparisons = []
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            diff = scores[keys[i]] - scores[keys[j]]
            main_comparisons.append(score_diff_to_ahp(diff))

    # --- 2. 收集子特徵偏好 (第二層次矩陣) ---
    print("\n" + "-"*40)
    print("進階偏好設定 (子特徵微調)")
    
    sub_inputs = {}
    
    # 報酬次矩陣 (CAGR vs Dividend)
    q_sub1 = "S1. 關於投資報酬，你更看重「資產長期的價格成長」還是「穩定發放的現金股息」？"
    opt_sub1 = {
        'A': {'text': '絕對看重資產成長 (不配息最好)', 'scores': [9.0]},
        'B': {'text': '偏好資產成長', 'scores': [3.0]},
        'C': {'text': '兩者同樣重要', 'scores': [1.0]},
        'D': {'text': '偏好穩定股息', 'scores': [1/3]},
        'E': {'text': '絕對看重現金股息 (落袋為安)', 'scores': [1/9]}
    }
    sub_inputs["Return_Main"] = ask_question(q_sub1, opt_sub1)

    # 風險次矩陣 (Volatility vs MaxDD)
    q_sub2 = "S2. 關於投資風險，你更討厭「每天價格上沖下洗」還是「經歷一次長達半年的深度虧損」？"
    opt_sub2 = {
        'A': {'text': '極度討厭每天上沖下洗 (波動率)', 'scores': [9.0]},
        'B': {'text': '比較討厭每天上沖下洗', 'scores': [3.0]},
        'C': {'text': '兩者一樣討厭', 'scores': [1.0]},
        'D': {'text': '比較討厭深度虧損', 'scores': [1/3]},
        'E': {'text': '極度討厭深度虧損 (最大回撤)', 'scores': [1/9]}
    }
    sub_inputs["Risk_Main"] = ask_question(q_sub2, opt_sub2)

    # 流動性次矩陣 (Volume vs AUM)
    q_sub3 = "S3. 關於流動性，你更看重 ETF 的「每日交易活絡度」還是「總體資產規模(不易下市)」？"
    opt_sub3 = {
        'A': {'text': '極度看重每日交易活絡度', 'scores': [9.0]},
        'B': {'text': '比較看重交易活絡度', 'scores': [3.0]},
        'C': {'text': '兩者同等重要', 'scores': [1.0]},
        'D': {'text': '比較看重總資產規模', 'scores': [1/3]},
        'E': {'text': '極度看重總資產規模', 'scores': [1/9]}
    }
    sub_inputs["Liquidity_Main"] = ask_question(q_sub3, opt_sub3)

    # 組裝成類別所需的 user_inputs 格式
    return {
        "Main": main_comparisons,
        "Sub": sub_inputs
    }


# ==========================================
# 主程式執行區塊
# ==========================================
if __name__ == "__main__":
    ahp = TwoLevel_AHP_Model()
    
    # 執行問卷前處理器，取得 AHP 矩陣所需的數值
    user_simulation = build_user_simulation()
    
    # 餵給你的演算法核心
    global_weights, main_cr = ahp.calculate_global_weights(user_simulation)
    
    # 建立要匯出的字典
    ahp_export_data = {
        "Consistency_Ratio_CR": main_cr,
        "Global_Weights": global_weights
    }

    # 存成 JSON 檔案 (確保資料夾存在)
    os.makedirs("json", exist_ok=True)
    output_filename = "json\\stage2_ahp_global_weights.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(ahp_export_data, f, ensure_ascii=False, indent=4)

    print(f"\n✅ 兩層級 AHP 權重已成功匯出至 {output_filename}")
    print("總權重和驗證：", round(sum(global_weights.values()), 4))

 🧠 智能理財引擎：投資屬性與偏好分析問卷

Q1. 若遇到全球股災，你最多能忍受多少帳面虧損？
  (A) 絕對不能虧損 (極度厭惡風險)
  (B) 虧損 10% 以內
  (C) 虧損 20% 左右
  (D) 虧損 30% 也能接受，長期會漲回來
  (E) 腰斬也不怕，危機就是轉機 (極度追求報酬)

Q2. 購買高單價商品時，你的消費習慣是？
  (A) 極度精打細算，到處比價找折扣
  (B) 稍微比價，不花太多時間
  (C) 東西好稍微貴一點無所謂
  (D) 只看品牌和頂規，不在乎價差

Q3. 你未來 1 到 3 年內臨時需要動用這筆資金的機率？
  (A) 極高，可能隨時變現救急
  (B) 有點可能，需保留彈性
  (C) 機率很低
  (D) 絕對不會動用，鎖死 10 年也沒差

Q4. 若要經營一門生意，你偏好？
  (A) 複合式商場，什麼都賣以分散風險
  (B) 幾樣主力商品搭配配件
  (C) 專賣店，專攻利基市場
  (D) 極致單一產品，要麼大賺要麼倒閉

Q5. 當新聞大肆報導 AI 科技爆發時，你的反應是？
  (A) 立刻跟進，跟著資金流賺趨勢財
  (B) 適度參與，不全盤投入
  (C) 不受影響，只看基本面
  (D) 新聞越熱我越不敢碰

----------------------------------------
進階偏好設定 (子特徵微調)

S1. 關於投資報酬，你更看重「資產長期的價格成長」還是「穩定發放的現金股息」？
  (A) 絕對看重資產成長 (不配息最好)
  (B) 偏好資產成長
  (C) 兩者同樣重要
  (D) 偏好穩定股息
  (E) 絕對看重現金股息 (落袋為安)

S2. 關於投資風險，你更討厭「每天價格上沖下洗」還是「經歷一次長達半年的深度虧損」？
  (A) 極度討厭每天上沖下洗 (波動率)
  (B) 比較討厭每天上沖下洗
  (C) 兩者一樣討厭
  (D) 比較討厭深度虧損
  (E) 極度討厭深度虧損 (最大回撤)

S3. 關於流動性，你更看重 ETF 的「每日交易活絡度」還是「總體資產規模(不易下市)」？
  (A) 極度看重每日交易活絡度
  (B) 比較看重交易活絡度
  (C) 兩者同等重要
  (D) 比較看重總資產規模
  (E) 極度看重總資產規模


In [1]:
import numpy as np
import json

class TwoLevel_AHP_Model:
    def __init__(self):
        # --- 第一層：主維度 ---
        self.main_criteria = [
            "Return_Main", 
            "Risk_Main", 
            "Cost_Main", 
            "Liquidity_Main", 
            "Diversity_Main", 
            "Sentiment_Main"
        ]
        
        # --- 第二層：子特徵 ---
        self.sub_criteria = {
            "Return_Main": ["Return_CAGR", "Return_Div"],
            "Risk_Main": ["Risk_Vol", "Risk_MaxDD"],
            "Liquidity_Main": ["Liq_Volume", "Liq_AUM"],
            "Cost_Main": ["Cost_ExpRatio"],       # 單一特徵，權重為 1
            "Diversity_Main": ["Div_Score"],      # 單一特徵，權重為 1
            "Sentiment_Main": ["FinBERT_score"]   # 單一特徵，權重為 1
        }
        
        # 定義不同矩陣大小對應的 Random Index (RI) 查表值
        self.RI_dict = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24}

    def _solve_matrix(self, comparisons, n):
        """通用的 AHP 矩陣求解器"""
        if n == 1:
            return np.array([1.0]), 0.0 # 單一特徵不需比較
            
        num_comparisons = (n * (n - 1)) // 2
        if len(comparisons) != num_comparisons:
            raise ValueError(f"維度為 {n} 的矩陣需要 {num_comparisons} 個成對比較值。")

        matrix = np.ones((n, n))
        idx = 0
        for i in range(n):
            for j in range(i + 1, n):
                val = comparisons[idx]
                matrix[i, j] = val
                matrix[j, i] = 1.0 / val
                idx += 1
                
        eigenvalues, eigenvectors = np.linalg.eig(matrix)
        max_idx = np.argmax(np.real(eigenvalues))
        max_eigenvalue = np.real(eigenvalues[max_idx])
        eigenvector = np.real(eigenvectors[:, max_idx])
        
        weights = eigenvector / np.sum(eigenvector)
        
        # 當 n<=2 時，CR 理論上恆為 0，且 RI 為 0 無法相除，因此直接回傳 CR=0
        if n <= 2:
            CR = 0.0
        else:
            CI = (max_eigenvalue - n) / (n - 1)
            CR = CI / self.RI_dict[n]
            
        return weights, CR

    def calculate_global_weights(self, user_inputs):
        """
        傳入使用者的問卷結果 (包含第一層與需要比較的第二層)。
        計算出最終 9 個子特徵的全局權重。
        """
        print("🚀 啟動兩層級 AHP (Two-Level AHP) 運算...")
        
        # 1. 求解第一層主維度權重
        print("\n[第一層：主維度求解]")
        main_weights, main_cr = self._solve_matrix(user_inputs["Main"], len(self.main_criteria))
        
        main_weight_dict = {crit: weight for crit, weight in zip(self.main_criteria, main_weights)}
        for crit, weight in main_weight_dict.items():
            print(f"  - {crit}: {weight*100:.2f}%")
        print(f"  >> 一致性比率 (CR): {main_cr:.4f}")
        
        if main_cr > 0.1:
            print("❌ 警告：主維度問卷存在邏輯矛盾 (CR > 0.1)！")
            
        # 2. 求解第二層子特徵權重並計算全局權重
        print("\n[第二層：子特徵局部權重與全局權重]")
        global_weights = {}
        
        for main_crit in self.main_criteria:
            subs = self.sub_criteria[main_crit]
            main_w = main_weight_dict[main_crit]
            
            # 如果該維度只有 1 個子特徵，局部權重為 1.0
            if len(subs) == 1:
                sub_name = subs[0]
                global_weights[sub_name] = main_w * 1.0
                print(f"  - {sub_name} (單一特徵) -> 全局權重: {global_weights[sub_name]*100:.2f}%")
                continue
                
            # 如果有多個子特徵，需要求解次矩陣
            sub_comparisons = user_inputs["Sub"].get(main_crit, [])
            local_weights, sub_cr = self._solve_matrix(sub_comparisons, len(subs))
            
            for i, sub_name in enumerate(subs):
                global_w = main_w * local_weights[i]
                global_weights[sub_name] = global_w
                print(f"  - {sub_name} (局部 {local_weights[i]*100:.1f}%) -> 全局權重: {global_w*100:.2f}%")
                
        return global_weights, main_cr

# ==========================================
# 模擬執行與 JSON 匯出區塊
# ==========================================
if __name__ == "__main__":
    ahp = TwoLevel_AHP_Model()
    
    # 模擬一位使用者的問卷填答結果
    
    user_simulation = {
        # 第一層：15 個比較值 (激進成長型範例)
        "Main": [7, 7, 5, 4, 2, 1, 1/4, 1/5, 1/7, 1/4, 1/5, 1/7, 1/2, 1/4, 1/3],
        
        # 第二層：各次矩陣的比較值
        "Sub": {
            # 報酬層面：極度看重資本增值 (CAGR) 勝過配息 (Div)
            "Return_Main": [9], 
            # 風險層面：覺得最大回撤 (MaxDD) 比日常波動 (Vol) 重要
            "Risk_Main": [1/3], 
            # 流動性層面：覺得日均成交量 (Volume) 與規模 (AUM) 同等重要
            "Liquidity_Main": [1] 
        }
    }
    '''
    user_simulation = {
        # 第一層：15 個比較值 
        # 邏輯：報酬(C1)與風險(C2)同等重要(1)，且兩者皆「極端重要(9)」於其他四個維度。其餘四者互相同等重要(1)。
        "Main": [
            1/10,   # 報酬 vs 風險 (同等重要)
            100000,   # 報酬 vs 成本 
            100000,   # 報酬 vs 流動 
            100000,   # 報酬 vs 分散 
            100000,   # 報酬 vs 情緒 
            1000000,   # 風險 vs 成本 
            1000000,   # 風險 vs 流動 
            1000000,   # 風險 vs 分散 
            1000000,   # 風險 vs 情緒 
            1,   # 成本 vs 流動 
            1,   # 成本 vs 分散 
            1,   # 成本 vs 情緒 
            1,   # 流動 vs 分散 
            1,   # 流動 vs 情緒 
            1    # 分散 vs 情緒 
        ],
        
        # 第二層：各次矩陣的比較值
        "Sub": {
            # 報酬層面：極端看重資本增值 (CAGR)，完全忽視配息
            "Return_Main": [1000000], 
            # 風險層面：極端看重日常波動 (Vol)，完全忽視最大回撤
            "Risk_Main": [1000000], 
            # 流動性層面：不重要，設為同等
            "Liquidity_Main": [1] 
        }
    }
    '''
    
    global_weights, main_cr = ahp.calculate_global_weights(user_simulation)
    
    # 建立要匯出的字典，與 Stage 3 的特徵名稱完全對齊
    ahp_export_data = {
        "Consistency_Ratio_CR": main_cr,
        "Global_Weights": global_weights
    }

    # 存成 JSON 檔案
    output_filename = "json\\stage2_ahp_global_weights.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(ahp_export_data, f, ensure_ascii=False, indent=4)

    print(f"\n✅ 兩層級 AHP 權重已成功匯出至 {output_filename}")
    print("總權重和驗證：", round(sum(global_weights.values()), 4))

🚀 啟動兩層級 AHP (Two-Level AHP) 運算...

[第一層：主維度求解]
  - Return_Main: 40.35%
  - Risk_Main: 3.55%
  - Cost_Main: 3.55%
  - Liquidity_Main: 9.65%
  - Diversity_Main: 14.17%
  - Sentiment_Main: 28.73%
  >> 一致性比率 (CR): 0.0414

[第二層：子特徵局部權重與全局權重]
  - Return_CAGR (局部 90.0%) -> 全局權重: 36.31%
  - Return_Div (局部 10.0%) -> 全局權重: 4.03%
  - Risk_Vol (局部 25.0%) -> 全局權重: 0.89%
  - Risk_MaxDD (局部 75.0%) -> 全局權重: 2.66%
  - Cost_ExpRatio (單一特徵) -> 全局權重: 3.55%
  - Liq_Volume (局部 50.0%) -> 全局權重: 4.83%
  - Liq_AUM (局部 50.0%) -> 全局權重: 4.83%
  - Div_Score (單一特徵) -> 全局權重: 14.17%
  - FinBERT_score (單一特徵) -> 全局權重: 28.73%

✅ 兩層級 AHP 權重已成功匯出至 json\stage2_ahp_global_weights.json
總權重和驗證： 1.0
